# Training Window Sensitivity

Compare four training-window strategies for the locked XGBoost-with-Elo pipeline:

| Method | OOF fold for test year Y | 2025 holdout fit |
|---|---|---|
| `extend_2015` | train [2015..Y-2], ES val Y-1, predict Y | train [2015..2023], ES val 2024, predict 2025 |
| `extend_2018` | train [2018..Y-2], ES val Y-1, predict Y | train [2018..2023], ES val 2024, predict 2025 |
| `roll_2yr`     | train [Y-2], ES val Y-1, predict Y       | train [2023], ES val 2024, predict 2025 |
| `roll_3yr`     | train [Y-3..Y-2], ES val Y-1, predict Y  | train [2022..2023], ES val 2024, predict 2025 |

All other hyperparameters and feature columns are held fixed (`final_model.XGB_PARAMS`, `FEAT_COLS`).

Trading benchmark on 2025 holdout uses the same configuration in every method:
- half-Kelly best config (`edge_min = 0.05`, `norm_min = 0.25`)
- \$5,000 starting bankroll
- sweep execution against historical Kalshi tape

**Outputs** written to `organized/outputs/`:
- `training_windows_logloss_summary.csv`
- `training_windows_logloss_per_fold.csv`
- `training_windows_trading_summary.csv`
- `training_windows_logloss_comparison.png`
- `training_windows_equity_curves.png`

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({'font.size': 11, 'axes.titlesize': 13, 'figure.dpi': 120})

ANALYSIS_DIR = Path('../..').resolve() / 'src' / 'srwnba' / 'analysis'
PIPELINE_DIR = Path('../..').resolve() / 'pipelines' / '05_modeling'
for d in (ANALYSIS_DIR, PIPELINE_DIR):
    if str(d) not in sys.path:
        sys.path.insert(0, str(d))

from outputs import save_fig, save_table
from walkforward import walk_forward_models, fit_holdout_models, summarize_predictions
from markets import build_kalshi_trading_index
from trading import (
    collect_entries, run_kelly_sweep, add_trade_returns, equity_by_payout,
)
from final_model import load_year, LABEL_COL

OOF_YEARS = list(range(2020, 2025))
HOLDOUT_YEAR = 2025
BANKROLL_REAL = 5000.0
BEST_EDGE_MIN = 0.05
BEST_NORM_MIN = 0.25

MODEL_SPEC = {'xgb_with_elo': {'type': 'xgb', 'use_bm': True}}

METHODS = {
    'extend_2015': {'kind': 'extend',  'start': 2015, 'label': 'Extend (2015→)',           'color': '#3498db'},
    'extend_2018': {'kind': 'extend',  'start': 2018, 'label': 'Extend (2018→)',           'color': '#2ecc71'},
    'roll_2yr':    {'kind': 'rolling', 'years': 2,    'label': 'Rolling 2-year',           'color': '#e67e22'},
    'roll_3yr':    {'kind': 'rolling', 'years': 3,    'label': 'Rolling 3-year',           'color': '#e74c3c'},
}
list(METHODS)

---
## 1. Walk-forward OOF (2020–2024) for each method

Each `(method, test_year)` fold runs the same nested early-stopping protocol but with the training-window start tied to the method:
- `extend_*`: fixed start year, growing fold-by-fold.
- `roll_Nyr`: start year = `test_year − N`, rolling fold-by-fold.

Internally we call the existing `walk_forward_models` once per fold with the appropriate `train_start`.

In [ ]:
def train_start_for(method_key, test_year):
    cfg = METHODS[method_key]
    if cfg['kind'] == 'extend':
        return cfg['start']
    return test_year - cfg['years']

def run_oof_for_method(method_key):
    rows = []
    for test_year in OOF_YEARS:
        ts = train_start_for(method_key, test_year)
        # walk_forward_models loads load_year(y) for y in range(ts, test_year-1)
        # then ES on test_year-1 and predicts test_year — we just call it with
        # a single-fold list so train_start is per-fold tunable.
        sub = walk_forward_models(
            MODEL_SPEC, oof_years=[test_year], train_start=ts, verbose=False,
        )
        sub['method'] = method_key
        sub['train_start'] = ts
        sub['test_year'] = test_year
        rows.append(sub)
    return pd.concat(rows, ignore_index=True)

oof_all = {}
for k in METHODS:
    print(f'OOF · {k}')
    oof_all[k] = run_oof_for_method(k)
    print(f'  rows: {len(oof_all[k]):,}  |  unique games: {oof_all[k]["game_id"].nunique()}')

In [ ]:
# Per-fold log-loss table (one row per method × test_year)
from sklearn.metrics import log_loss

fold_rows = []
for k, df in oof_all.items():
    for test_year in OOF_YEARS:
        sub = df[df['season'] == test_year]
        ll = log_loss(sub['home_win'], np.clip(sub['pred_prob'], 1e-7, 1-1e-7))
        fold_rows.append({
            'method': k, 'test_year': test_year,
            'train_start': int(sub['train_start'].iloc[0]),
            'log_loss': float(ll), 'n_games': int(len(sub)),
        })
fold_tbl = pd.DataFrame(fold_rows)
save_table(fold_tbl, 'training_windows_logloss_per_fold')
fold_tbl

---
## 2. 2025 holdout fit for each method

In [ ]:
holdout_preds = {}
holdout_starts = {}
for k, cfg in METHODS.items():
    ts = (cfg['start'] if cfg['kind'] == 'extend' else HOLDOUT_YEAR - cfg['years'])
    holdout_starts[k] = ts
    df, _ = fit_holdout_models(
        MODEL_SPEC, holdout_year=HOLDOUT_YEAR, train_start=ts, verbose=False,
    )
    df['method'] = k
    df['train_start'] = ts
    holdout_preds[k] = df
    n = df[df['model'] == 'xgb_with_elo'].shape[0]
    ll = log_loss(
        df.loc[df['model']=='xgb_with_elo','home_win'],
        np.clip(df.loc[df['model']=='xgb_with_elo','pred_prob'], 1e-7, 1-1e-7),
    )
    print(f'  {k:<12s}  train_start={ts}  n={n}  2025 LL={ll:.4f}')

---
## 3. Log-loss summary across methods

In [ ]:
rows = []
for k, cfg in METHODS.items():
    sub_oof = fold_tbl[fold_tbl['method'] == k]
    mean_ll = float(sub_oof['log_loss'].mean())
    h_df = holdout_preds[k]
    h_sub = h_df[h_df['model'] == 'xgb_with_elo']
    holdout_ll = float(log_loss(h_sub['home_win'], np.clip(h_sub['pred_prob'], 1e-7, 1-1e-7)))
    rows.append({
        'method': k, 'label': cfg['label'],
        'train_start_2025': holdout_starts[k],
        'oof_mean_log_loss': mean_ll,
        'oof_min_log_loss': float(sub_oof['log_loss'].min()),
        'oof_max_log_loss': float(sub_oof['log_loss'].max()),
        'holdout_log_loss_2025': holdout_ll,
    })
ll_summary = pd.DataFrame(rows)
save_table(ll_summary, 'training_windows_logloss_summary')
ll_summary

In [ ]:
# Per-fold log-loss line plot + 2025 holdout markers
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for k, cfg in METHODS.items():
    sub = fold_tbl[fold_tbl['method'] == k].sort_values('test_year')
    ax.plot(sub['test_year'], sub['log_loss'], marker='o', linewidth=2,
            color=cfg['color'], label=cfg['label'])
ax.set_xlabel('Test year (OOF fold)'); ax.set_ylabel('Log loss')
ax.set_title('Per-Fold OOF Log Loss by Training Window', fontweight='bold')
ax.set_xticks(OOF_YEARS); ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

ax = axes[1]
x = np.arange(len(METHODS)); w = 0.38
labels = [cfg['label'] for cfg in METHODS.values()]
colors = [cfg['color'] for cfg in METHODS.values()]
ax.bar(x - w/2, ll_summary['oof_mean_log_loss'], width=w,
       color=colors, alpha=0.55, edgecolor='white', label='Mean OOF (2020–2024)')
ax.bar(x + w/2, ll_summary['holdout_log_loss_2025'], width=w,
       color=colors, alpha=1.0, edgecolor='white', label='2025 holdout')
for xi, (a, b) in enumerate(zip(ll_summary['oof_mean_log_loss'],
                                 ll_summary['holdout_log_loss_2025'])):
    ax.text(xi - w/2, a + 0.001, f'{a:.4f}', ha='center', fontsize=8)
    ax.text(xi + w/2, b + 0.001, f'{b:.4f}', ha='center', fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9, rotation=15, ha='right')
ax.set_ylabel('Log loss')
ax.set_title('Mean OOF vs 2025 Holdout Log Loss', fontweight='bold')
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
save_fig(fig, 'training_windows_logloss_comparison')
plt.show()

---
## 4. 2025 holdout trading — half-Kelly sweep at \$5,000

For each method we:
1. Build 2025 signals using that method's `xgb_with_elo` predictions.
2. Reuse the same Kalshi market index (candles, settlement times, historical tape).
3. Run `run_kelly_sweep` with edge ≥ 0.05, norm ≥ 0.25, half-Kelly fraction = 0.5, \$5k bankroll.
4. Report total return, max drawdown, mean per-trade log return, per-trade Sharpe = mean/std of `log_ret`.

In [ ]:
test_2025 = (
    load_year(HOLDOUT_YEAR).dropna(subset=[LABEL_COL, 'base_margin'])
    [['game_id','game_ts','game_date','home_team_id','away_team_id', LABEL_COL,'p_elo']]
    .copy()
)
test_2025['game_ts'] = pd.to_datetime(test_2025['game_ts'], utc=True)
test_2025['game_date'] = pd.to_datetime(test_2025['game_date'])

trading_results = {}
trade_frames = {}
for k, cfg in METHODS.items():
    h_sub = holdout_preds[k]
    h_sub = h_sub[h_sub['model']=='xgb_with_elo'][['game_id','pred_prob']].rename(
        columns={'pred_prob': 'p_full_model'},
    )
    signals = test_2025.merge(h_sub, on='game_id')
    idx = build_kalshi_trading_index(signals, pred_cols=('p_full_model','p_elo'))
    ents = collect_entries(
        idx['ticker_info'], idx['pretip'], 'p_full_model',
        BEST_EDGE_MIN, BEST_NORM_MIN, 'half_life',
    )
    raw = run_kelly_sweep(ents, 0.5, idx['wt'], BANKROLL_REAL)
    if not raw:
        trading_results[k] = None
        continue
    tdf = add_trade_returns(pd.DataFrame(raw))
    eq  = equity_by_payout(tdf, bankroll_init=BANKROLL_REAL, ts_col='game_ts')
    fb_engine = float(tdf['bankroll'].iloc[-1])
    fb_payout = float(eq['display_bankroll'].iloc[-1])
    dd_engine = float((tdf['bankroll'].cummax() - tdf['bankroll']).max())
    mean_log_ret = float(tdf['log_ret'].mean())
    std_log_ret  = float(tdf['log_ret'].std())
    sharpe = mean_log_ret / std_log_ret if std_log_ret > 0 else float('nan')
    trading_results[k] = {
        'method':            k,
        'label':             cfg['label'],
        'n_trades':          int(len(tdf)),
        'hit_rate':          float(tdf['won'].mean()),
        'total_return':      (fb_engine - BANKROLL_REAL) / BANKROLL_REAL,
        'final_bankroll':    fb_engine,
        'final_bankroll_payout_view': fb_payout,
        'max_drawdown':      dd_engine,
        'mean_log_return':   mean_log_ret,
        'std_log_return':    std_log_ret,
        'sharpe_per_trade':  sharpe,
        'mean_fill_rate':    float(tdf['fill_pct'].mean()),
    }
    trade_frames[k] = (tdf, eq)

trading_summary = pd.DataFrame([r for r in trading_results.values() if r is not None])
save_table(trading_summary, 'training_windows_trading_summary')
trading_summary

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
for k, (tdf, eq) in trade_frames.items():
    cfg = METHODS[k]
    r = trading_results[k]
    ax.step(eq['game_ts'], eq['display_bankroll'], where='post',
            color=cfg['color'], linewidth=1.7,
            label=f"{cfg['label']} ({r['total_return']:.0%}, n={r['n_trades']})")
ax.axhline(BANKROLL_REAL, color='gray', linestyle=':', alpha=0.6,
           label=f'Starting bankroll (${BANKROLL_REAL:,.0f})')
ax.set_xlabel('Settlement date'); ax.set_ylabel('Bankroll ($)')
ax.set_title('2025 Half-Kelly Sweep Equity Curves by Training Window',
             fontweight='bold')
ax.legend(fontsize=9, loc='upper left')
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b'))
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
save_fig(fig, 'training_windows_equity_curves')
plt.show()

---
## 5. Notes on interpretation

- The mean OOF log loss averages five folds (2020–2024). Rolling windows have far less training data per fold, so OOF differences pick up both the bias from a smaller sample and the variance reduction from a more recent regime.
- The 2025 holdout log loss is a single point estimate; with only 310 games it is sample-noise-dominated, and small differences across methods should be read alongside the OOF means.
- Trading metrics use the same threshold pair (`edge ≥ 0.05, norm ≥ 0.25`) and the same Kalshi market data for all four methods, so any divergence comes only from how each method's predictions induce a different trade set / sizing path.
- Sharpe here is per-trade `mean(log_ret) / std(log_ret)` — no annualization, because the trade frequency itself differs across methods. For across-method comparison this is the right scale.
- `final_bankroll` reports the engine bankroll (entry-time compounding); `final_bankroll_payout_view` is the same trades in settlement-time order via `equity_by_payout`. Both should agree to within rounding.
- `extend_2018` is interesting because injury data only becomes reliably populated from 2018 onward — it's a fairer test of the locked feature design than `extend_2015`, which includes early seasons where a couple of injury features are essentially zero.